[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C46_Graph_ML_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零实现** 每一种 GNN 层，再在**小图**上 **对拍 / 验证收敛**。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子建立「图 = 邻接矩阵 + 特征矩阵，一步传播 = `A @ X`」的核心直觉；③ 立下全课的纪律——**置换不变 + 对拍 + 可复现**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` / `networkx` 可选（画图 / 取 Karate club），缺了不影响课程。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
for opt in ['matplotlib', 'networkx']:
    try:
        m = __import__(opt); print(opt, getattr(m, '__version__', '?'), '(可选)')
    except Exception:
        print(opt, '未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 一张图就是 (邻接矩阵 A, 特征矩阵 X)

我们用一个 5 节点的小无向图建立最基本的表示。约定：`A[i,j]=1` 表示节点 i,j 相连；无向图 `A` 对称；`X` 的第 i 行是节点 i 的特征。

图：0-1, 1-2, 2-3, 3-4, 4-0（一个五元环）再加一条对角 1-3。

In [ ]:
import numpy as np

edges = [(0,1),(1,2),(2,3),(3,4),(4,0),(1,3)]
n = 5
A = np.zeros((n, n))
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0          # 无向：对称
print('邻接矩阵 A =\n', A)
# 度 = 每行之和
deg = A.sum(axis=1)
print('度向量 deg =', deg)
assert np.allclose(A, A.T), '无向图邻接矩阵必须对称'
assert deg.tolist() == [2,3,2,3,2], '度数核对：节点1,3 各 3 条边'
print('✅ 对称性与度数核对通过')

## 3 · 一步「传播」就是 A @ X：把邻居特征加起来

GNN 的灵魂操作：用每个节点的**邻居特征之和**来刻画它。这恰好就是矩阵乘 `A @ X`——
`(A @ X)[i] = Σ_j A[i,j] * X[j] = 邻居 j 的特征之和`。这是后面所有消息传递的最朴素形态。

In [ ]:
# 给每个节点一个 2 维特征
X = np.array([[1.,0.],[0.,1.],[1.,1.],[0.,0.],[2.,1.]])
agg = A @ X                       # 每个节点 <- 邻居特征之和
print('A @ X (邻居特征之和) =\n', agg)
# 手算验证节点 0：邻居是 1 和 4 -> X[1]+X[4] = [0,1]+[2,1] = [2,2]
assert np.allclose(agg[0], X[1] + X[4]), '节点0应等于邻居1,4特征之和'
# 节点 3：邻居 2,4,1 -> X[2]+X[4]+X[1]
assert np.allclose(agg[3], X[2] + X[4] + X[1])
print('✅ A @ X 正是「邻居特征求和」——一切消息传递的起点')

## 4 · 立纪律 (一)：置换不变 / 等变

图的节点编号是**任意的**。把节点重新编号（用置换矩阵 P），图还是同一张图。一个合格的图运算必须**置换等变**：
重排输入节点，输出也按同样方式重排。我们来验证「一步传播」满足这条对称性。

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(n)              # 一个随机的节点重编号
P = np.eye(n)[perm]                    # 置换矩阵：P @ v 把 v 按 perm 重排

# 重排后的图：A' = P A Pᵀ, X' = P X
A_perm = P @ A @ P.T
X_perm = P @ X
out_then_perm = P @ (A @ X)            # 先在原图传播，再重排输出
perm_then_out = A_perm @ X_perm        # 先重排，再在新图传播
assert np.allclose(out_then_perm, perm_then_out), '一步传播必须置换等变'
print('✅ 置换等变成立：A@X 不依赖节点编号（先排后算 == 先算后排）')
print('  这是几何深度学习对图模型的硬约束——本课每一层都满足它。')

## 5 · 立纪律 (二)：对拍——稀疏实现 vs 稠密实现

真实图很稀疏，框架用「逐边/逐邻居」的稀疏聚合而非稠密矩阵乘。我们写一个**稀疏版**传播（只遍历真实边），和稠密版 `A @ X` **对拍**。这就是全课的工作流：写实现 → 对拍可信参考 → assert 兜底。

In [ ]:
def propagate_sparse(edges, X, n, add_self=False):
    '''逐边聚合：每条无向边把两端互相加过去。等价于 A @ X（稀疏实现）。'''
    out = np.zeros_like(X)
    for i, j in edges:
        out[i] += X[j]
        out[j] += X[i]        # 无向边：双向
    if add_self:
        out += X
    return out

got = propagate_sparse(edges, X, n)
ref = A @ X
assert np.allclose(got, ref), '稀疏逐边聚合应等于稠密 A@X'
print('✅ 对拍通过：稀疏(逐边) == 稠密(A@X)')
print('  真实框架走稀疏路径(省内存/算力)，但结果必须和稠密参考一致——这就是对拍。')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成小函数，后面每个模块都用它判定「我的实现 == 可信参考」。它是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_allclose(name, got, ref, atol=1e-8):
    '''对拍：被测实现结果 vs 可信参考。返回是否一致并打印最大误差。'''
    got = np.asarray(got, dtype=float); ref = np.asarray(ref, dtype=float)
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：行归一化传播 D^{-1} A X 对拍两种写法
Dinv = np.diag(1.0 / deg)
way1 = Dinv @ (A @ X)
way2 = (Dinv @ A) @ X
check_allclose('D^-1 A X 结合律', way1, way2)
print('\n这就是全课的工作流：写实现 -> 对拍可信参考 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个图运算都满足置换等变、都用 `np.allclose` 对拍可信参考、都固定随机种子可复现；结构正确则数值一致，数值一致则逻辑可迁移到 PyG/DGL。

**接下来六个模块**：01 图与谱基础 → 02 消息传递与 GCN → 03 GAT 与 GraphSAGE → 04 图 Transformer → 05 可扩展与应用。每一步都建立在前一步之上。

下一站：**模块 01 · 图与谱基础** —— 把图翻译成线性代数，从拉普拉斯的谱里读出图的结构。